In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
NOTEBOOK = "/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/notebooks/02_jobs/20_ingest_run"
JOB_NAME = "oil-shock-ingest"

existing = [j for j in w.jobs.list(name=JOB_NAME)]
if existing:
    job_id = existing[0].job_id
    print(f"Job already exists: {job_id}")
else:
    job = w.jobs.create(
        name=JOB_NAME,
        tasks=[jobs.Task(
            task_key="ingest",
            notebook_task=jobs.NotebookTask(notebook_path=NOTEBOOK, source=jobs.Source.WORKSPACE),
            timeout_seconds=3600,
        )],
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 7,21 * * ?",
            timezone_id="Asia/Dubai",
            pause_status=jobs.PauseStatus.PAUSED,
        ),
        email_notifications=jobs.JobEmailNotifications(on_failure=["zulhaq2@gmail.com"]),
        max_concurrent_runs=1,
    )
    job_id = job.job_id
    print(f"Created job {JOB_NAME}: {job_id} (schedule paused)")

run = w.jobs.run_now(job_id=job_id)
print(f"Test run started: run_id {run.run_id}")
print("Watch it under Jobs & Pipelines > oil-shock-ingest > Runs")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
job_id = 970079484069909

w.jobs.update(job_id=job_id, new_settings=jobs.JobSettings(
    schedule=jobs.CronSchedule(
        quartz_cron_expression="0 0 7,21 * * ?",
        timezone_id="Asia/Dubai",
        pause_status=jobs.PauseStatus.UNPAUSED,
    )
))

s = w.jobs.get(job_id).settings.schedule
print(f"Schedule: {s.quartz_cron_expression} ({s.timezone_id}) — {s.pause_status.value}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
job_id = 970079484069909
NB = "/Users/zulhaq2@gmail.com/oil-shock-2026-databricks/notebooks/02_jobs"

w.jobs.reset(job_id=job_id, new_settings=jobs.JobSettings(
    name="oil-shock-ingest",
    tasks=[
        jobs.Task(
            task_key="ingest",
            notebook_task=jobs.NotebookTask(notebook_path=f"{NB}/20_ingest_run", source=jobs.Source.WORKSPACE),
            timeout_seconds=3600,
        ),
        jobs.Task(
            task_key="dbt_build",
            depends_on=[jobs.TaskDependency(task_key="ingest")],
            run_if=jobs.RunIf.ALL_DONE,
            notebook_task=jobs.NotebookTask(notebook_path=f"{NB}/22_dbt_build", source=jobs.Source.WORKSPACE),
            timeout_seconds=3600,
        ),
    ],
    schedule=jobs.CronSchedule(quartz_cron_expression="0 0 7,21 * * ?", timezone_id="Asia/Dubai",
                               pause_status=jobs.PauseStatus.UNPAUSED),
    email_notifications=jobs.JobEmailNotifications(on_failure=["zulhaq2@gmail.com"]),
    max_concurrent_runs=1,
))

s = w.jobs.get(job_id).settings
print("Tasks:", [t.task_key for t in s.tasks], "| Schedule:", s.schedule.quartz_cron_expression, s.schedule.pause_status.value)

run = w.jobs.run_now(job_id=job_id)
print("Test run started:", run.run_id)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

JOB_ID = 970079484069909
CHAIN = ["45_anomaly_flags", "44_scenarios", "40_event_impact", "41_driver_attribution",
         "43_inventory_nowcast", "46_eia_range_refresh"]

w = WorkspaceClient()
settings = w.jobs.get(JOB_ID).settings
base = next(t for t in settings.tasks if t.task_key == "dbt_build")
base_path = base.notebook_task.notebook_path
existing = {t.task_key for t in settings.tasks}

prev = "dbt_build"
for i, nb in enumerate(CHAIN):
    key = "ml_" + nb.split("_", 1)[1]
    if key not in existing:
        settings.tasks.append(jobs.Task(
            task_key=key,
            depends_on=[jobs.TaskDependency(task_key=prev)],
            run_if=jobs.RunIf.ALL_SUCCESS if i == 0 else jobs.RunIf.ALL_DONE,
            notebook_task=jobs.NotebookTask(
                notebook_path=base_path.replace("02_jobs/22_dbt_build", f"04_ml/{nb}"),
                source=base.notebook_task.source),
            environment_key=base.environment_key,
            timeout_seconds=1800))
    prev = key

w.jobs.reset(JOB_ID, new_settings=settings)
for t in w.jobs.get(JOB_ID).settings.tasks:
    deps = [d.task_key for d in (t.depends_on or [])]
    print(f"{t.task_key:<24} after {deps} | run_if {t.run_if} | {t.notebook_task.notebook_path}")

In [0]:
# Remove Events, Drivers and Anomalies tasks from the job (no deletes of data or notebooks)
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

w = WorkspaceClient()
JOB_ID = 970079484069909
DROP = {"ml_event_impact", "ml_driver_attribution", "ml_anomaly_flags"}

s = w.jobs.get(JOB_ID).settings
by_key = {t.task_key: t for t in s.tasks}
missing = DROP - set(by_key)
if missing:
    print("Not in job (already removed?):", sorted(missing))

def kept_parents(key):
    out = []
    for d in (by_key[key].depends_on or []):
        out += kept_parents(d.task_key) if d.task_key in DROP else [d.task_key]
    return sorted(set(out))

print("BEFORE:")
for t in s.tasks:
    print(f"   {t.task_key:<24} after {[d.task_key for d in (t.depends_on or [])]}")

new_tasks = []
for t in s.tasks:
    if t.task_key in DROP:
        continue
    parents = kept_parents(t.task_key)
    t.depends_on = [jobs.TaskDependency(task_key=p) for p in parents] or None
    new_tasks.append(t)
s.tasks = new_tasks
w.jobs.reset(job_id=JOB_ID, new_settings=s)

print("AFTER:")
for t in w.jobs.get(JOB_ID).settings.tasks:
    print(f"   {t.task_key:<24} after {[d.task_key for d in (t.depends_on or [])]}")
print("DONE - paste this output back to Claude")